In [1]:
%load_ext autoreload
%autoreload 2

import meles
import numpy as np
import pandas as pd

## Download YoutubeFaces

First, we download the YoutubeFaces dataset by requesting the credentials and accessing the [downlaod page](http://www.cslab.openu.ac.il/download/wolftau/). To save on system resources, we will only download the aligned video frames (i.e., `aligned_images_DB.tar.gz`) which are cropped and without audio.


In [ ]:
# !curl -u username:password -L -C - -O http://www.cslab.openu.ac.il/download/wolftau/aligned_images_DB.tar.gz

Next, we need to place the extract the tarball into the `/data/ytfaces` directory. We will use the same directory structure for our dataset.

In [4]:
import json
from pathlib import Path

# Define the ytfaces data directory
ytfaces_dir = Path("../../data/ytfaces")
output_file = Path("../../data/ytfaces_structure.json")


# Function to explore directory structure for ytfaces dataset
def explore_ytfaces_structure(root_path):
    """
    Explore ytfaces directory structure: subject_name/video_number/frames.jpg
    Returns a slim structure with just paths
    """
    structure = {}

    if not root_path.exists():
        return structure

    # Iterate through subjects
    for subject_dir in sorted(root_path.iterdir()):
        if not subject_dir.is_dir():
            continue

        subject_name = subject_dir.name
        structure[subject_name] = {}

        # Iterate through videos for each subject
        for video_dir in sorted(subject_dir.iterdir()):
            if not video_dir.is_dir():
                continue

            video_number = video_dir.name
            structure[subject_name][video_number] = []

            # Collect all frame files
            for frame_file in sorted(video_dir.iterdir()):
                if frame_file.is_file() and frame_file.suffix.lower() in [
                    ".jpg",
                    ".jpeg",
                    ".png",
                ]:
                    structure[subject_name][video_number].append(frame_file.name)

    return structure


# Explore the directory structure
if ytfaces_dir.exists():
    dataset_structure = explore_ytfaces_structure(ytfaces_dir)

    # Save to JSON file
    with open(output_file, "w") as f:
        json.dump(dataset_structure, f, indent=2)

    print(f"Directory structure saved to: {output_file}")
    print(f"\nDataset overview:")
    print(f"  Total subjects: {len(dataset_structure)}")

    # Show sample of first subject
    if dataset_structure:
        first_subject = list(dataset_structure.keys())[0]
        print(f"\nExample - Subject '{first_subject}':")
        print(f"  Videos: {len(dataset_structure[first_subject])}")
        if dataset_structure[first_subject]:
            first_video = list(dataset_structure[first_subject].keys())[0]
            print(
                f"  Frames in video '{first_video}': {len(dataset_structure[first_subject][first_video])}"
            )
else:
    print(f"Directory not found: {ytfaces_dir}")

Directory structure saved to: ../../data/ytfaces_structure.json

Dataset overview:
  Total subjects: 1595

Example - Subject 'AJ_Cook':
  Videos: 2
  Frames in video '0': 79


In [5]:
# Display summary statistics
def get_ytfaces_stats(structure):
    """Calculate statistics about the ytfaces dataset structure"""
    stats = {"total_subjects": len(structure), "total_videos": 0, "total_frames": 0}

    for subject, videos in structure.items():
        stats["total_videos"] += len(videos)
        for video, frames in videos.items():
            stats["total_frames"] += len(frames)

    return stats


if dataset_structure:
    stats = get_ytfaces_stats(dataset_structure)
    print("Dataset Statistics:")
    print(f"  Total subjects: {stats['total_subjects']}")
    print(f"  Total videos: {stats['total_videos']}")
    print(f"  Total frames: {stats['total_frames']}")
    print(
        f"  Average videos per subject: {stats['total_videos'] / stats['total_subjects']:.1f}"
    )
    print(
        f"  Average frames per video: {stats['total_frames'] / stats['total_videos']:.1f}"
    )

Dataset Statistics:
  Total subjects: 1595
  Total videos: 3425
  Total frames: 621126
  Average videos per subject: 2.1
  Average frames per video: 181.4


Next, we read the structure of the dataset and convert it into a pandas dataframe for easier manipulation and analysis. This will allow us to perform various operations such as filtering, sorting, and aggregating data based on different criteria.

In [6]:
# TODO implement by flattening the JSON structure. The frames are in alphabetical order so its fine.